# Linraries

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Dataset

In [16]:
df = pd.DataFrame({
    "user_id": [22, 35, 28, 40, 19, 33, 30, 27, 45, 23],
    "user_gender": ['M', 'F', 'M', 'F', 'M', 'F', 'M', 'M', 'M', 'F'],
    "user_country": ['US', 'IN', 'UK', 'US', 'IN', 'FR', 'DE', 'IN', 'US', 'UK'],
    "ad_id": ['A1', np.nan, 'A1', 'A3', 'A2', 'A1', 'A4', 'A3', 'A2', 'A1'],
    "ad_type": ['banner', 'video', 'banner', np.nan, 'video', 'banner', 'native', 'video', 'banner', 'native'],
    "bid_price": [0.25, 0.90, 0.40, 0.55, 1.10, 0.30, 0.75, 0.95, np.nan, 0.60],
    "device": ['mobile', 'desktop', 'mobile', 'tablet', 'mobile', 'desktop', 'mobile', 'tablet', 'mobile', 'desktop'],
    "hour": [10, 18, 14, 20, 9, 16, 12, 19, 11, 21],
    "day_of_week": [1, 4, 2, 0, 3, 1, 6, 2, 4, 1],
    "prev_click": [0, 1, 0, 2, 0, 1, 1, 0, 0, 2],
    "impression": [3, 5, 2, 6, 4, 5, 3, 4, 2, 7],
    "clicked": [0, 1, 0, 1, 0, 1, 0, 0, 0, 1]
})

print(df)

   user_id user_gender user_country ad_id ad_type  bid_price   device  hour  \
0       22           M           US    A1  banner       0.25   mobile    10   
1       35           F           IN   NaN   video       0.90  desktop    18   
2       28           M           UK    A1  banner       0.40   mobile    14   
3       40           F           US    A3     NaN       0.55   tablet    20   
4       19           M           IN    A2   video       1.10   mobile     9   
5       33           F           FR    A1  banner       0.30  desktop    16   
6       30           M           DE    A4  native       0.75   mobile    12   
7       27           M           IN    A3   video       0.95   tablet    19   
8       45           M           US    A2  banner        NaN   mobile    11   
9       23           F           UK    A1  native       0.60  desktop    21   

   day_of_week  prev_click  impression  clicked  
0            1           0           3        0  
1            4           1    

# Q1

In [17]:
# select columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# fill numerical NaNs with mean
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# fill categorical NaNs with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


# Q2

In [18]:
# Separate target & features
y = df['clicked']
X = df.drop(columns=['clicked'], axis=1)
X

,user_id,user_gender,user_country,ad_id,ad_type,bid_price,device,hour,day_of_week,prev_click,impression
0,22,M,US,A1,banner,0.250000,mobile,10,1,0,3
1,35,F,IN,A1,video,0.900000,desktop,18,4,1,5
2,28,M,UK,A1,banner,0.400000,mobile,14,2,0,2
3,40,F,US,A3,banner,0.550000,tablet,20,0,2,6
4,19,M,IN,A2,video,1.100000,mobile,9,3,0,4
5,33,F,FR,A1,banner,0.300000,desktop,16,1,1,5
6,30,M,DE,A4,native,0.750000,mobile,12,6,1,3
7,27,M,IN,A3,video,0.950000,tablet,19,2,0,4
8,45,M,US,A2,banner,0.644444,mobile,11,4,0,2
9,23,F,UK,A1,native,0.600000,desktop,21,1,2,7


# Q3

In [19]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Q4

In [20]:
# Encode required columns
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train)
X_test_enc = enc.transform(X_test)

#Q5

In [21]:
# Build and train SGD classifier model
sgd_model = SGDClassifier(loss='log_loss', penalty='elasticnet', l1_ratio=0.5, max_iter=1000, random_state=42)
sgd_model.fit(X_train_enc, y_train)

# Predict
y_pred = sgd_model.predict(X_test_enc)
y_pred_proba = sgd_model.predict_proba(X_test_enc)[:, 1]

# Print evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-Score: 1.0
ROC-AUC Score: 1.0


# Q6

In [22]:
# Find best parameters using GridSearchCV
param_grid = {
    'loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge'],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'l1_ratio': [0.15, 0.3, 0.5, 0.7, 0.85],
    'max_iter': [500, 1000, 1500, 2000]
}

grid_search = GridSearchCV(SGDClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_enc, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best ROC-AUC Score:", grid_search.best_score_)

Best Parameters: {'l1_ratio': 0.15, 'loss': 'hinge', 'max_iter': 500, 'penalty': 'l1'}
Best ROC-AUC Score: 1.0


# Q7

In [23]:
# get feature names after encoding
feature_names = enc.get_feature_names_out()

# get coefficient array (for binary classification)
coefs = grid_search.best_estimator_.coef_[0]

# sort features by absolute importance
sorted_idx = np.argsort(np.abs(coefs))

# least important 10 features
print("Least important features:")
print(feature_names[sorted_idx[:10]])

# most important 10 features
print("\nMost important features:")
print(feature_names[sorted_idx[-10:]])


Least important features:
['user_id_27' 'user_id_23' 'user_id_30' 'user_id_28' 'user_country_UK'
 'user_country_US' 'ad_id_A1' 'user_country_DE' 'bid_price_0.4'
 'bid_price_0.6']

Most important features:
['user_country_IN' 'ad_id_A2' 'ad_type_video' 'impression_4'
 'day_of_week_3' 'hour_9' 'user_gender_F' 'device_mobile' 'user_gender_M'
 'prev_click_0']
